# Russian Emotion Analysis for OCR Output

Self-contained notebook for the rule-based emotion module.
It includes:
- character trigram correction
- optional `pymorphy3` lemmatization
- `RusEmoLex` loading
- rule-based emotion scoring for OCR text


## External resources

The notebook keeps all logic inside the `ipynb`, but it can use external resources:
- `data/lexicons/RusEmoLex.csv`
- optional cached `data/lexicons/wordfreq_ru.tsv`

If these resources are absent, the notebook falls back to a tiny demo lexicon and vocabulary.


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
import csv
import math
import re

ROOT = Path.cwd()
LEXICON_PATH = ROOT / 'data' / 'lexicons' / 'RusEmoLex.csv'
WORDFREQ_FREQ_PATH = ROOT / 'data' / 'lexicons' / 'wordfreq_ru.tsv'
WORDFREQ_LANGUAGE = 'ru'
WORDFREQ_WORDLIST = 'large'
WORDFREQ_MAX_WORDS = 200_000
WORDFREQ_MIN_ZIPF = 2.5

RESOURCE_INFO = [
    {
        'resource': 'wordfreq wheel',
        'size': '56.8 MB',
        'purpose': 'Russian word frequencies for trigram reranking',
        'source': 'https://pypi.org/project/wordfreq/',
    },
    {
        'resource': 'RusEmoLex.csv',
        'size': '119 KB',
        'purpose': 'Emotion lexicon for rule-based scoring',
        'source': 'https://github.com/nl-pi/rusemolex/blob/main/RusEmoLex.csv',
    },
    {
        'resource': 'pymorphy3 wheel',
        'size': '53.9 kB',
        'purpose': 'Lemmatizer code',
        'source': 'https://pypi.org/project/pymorphy3/',
    },
    {
        'resource': 'pymorphy3-dicts-ru wheel',
        'size': '8.4 MB',
        'purpose': 'Russian morphology dictionaries',
        'source': 'https://pypi.org/project/pymorphy3-dicts-ru/',
    },
]

for item in RESOURCE_INFO:
    print(item)

print()
for candidate in [LEXICON_PATH, WORDFREQ_FREQ_PATH]:
    print(f'{candidate}:', 'exists' if candidate.exists() else 'missing')

try:
    import pymorphy3  # noqa: F401
    PYMORPHY3_AVAILABLE = True
except ImportError:
    PYMORPHY3_AVAILABLE = False

try:
    from wordfreq import iter_wordlist, zipf_frequency  # noqa: F401
    WORDFREQ_AVAILABLE = True
except ImportError:
    WORDFREQ_AVAILABLE = False

print('\npymorphy3 available:', PYMORPHY3_AVAILABLE)
print('wordfreq available:', WORDFREQ_AVAILABLE)


## Embedded implementation

This cell contains the standalone trigram corrector, RusEmoLex loader, and analyzer.


In [ ]:
EMOTION_LABEL_MAP = {
    'joy': 'joy',
    '\u0440\u0430\u0434\u043e\u0441\u0442\u044c': 'joy',
    'happiness': 'joy',
    'sadness': 'sadness',
    '\u0433\u0440\u0443\u0441\u0442\u044c': 'sadness',
    '\u043f\u0435\u0447\u0430\u043b\u044c': 'sadness',
    'fear': 'fear',
    '\u0441\u0442\u0440\u0430\u0445': 'fear',
    'anger': 'anger',
    '\u0437\u043b\u043e\u0441\u0442\u044c': 'anger',
    '\u0433\u043d\u0435\u0432': 'anger',
    'surprise': 'surprise',
    '\u0443\u0434\u0438\u0432\u043b\u0435\u043d\u0438\u0435': 'surprise',
    'disgust': 'disgust',
    '\u043e\u0442\u0432\u0440\u0430\u0449\u0435\u043d\u0438\u0435': 'disgust',
    'neutral': 'neutral',
    '\u043d\u0435\u0439\u0442\u0440\u0430\u043b\u044c\u043d\u044b\u0439': 'neutral',
    'none': 'unassigned',
    'unassigned': 'unassigned',
}

HEADER_ALIASES = {
    'word': 'term',
    '\u0441\u043b\u043e\u0432\u043e': 'term',
    'lemma': 'term',
    '\u043b\u0435\u043c\u043c\u0430': 'term',
    'pos': 'part_of_speech',
    'part of speech': 'part_of_speech',
    'part_of_speech': 'part_of_speech',
    '\u0447\u0430\u0441\u0442\u044c \u0440\u0435\u0447\u0438': 'part_of_speech',
    '\u043a\u043b\u0430\u0441\u0441': 'emotion',
    'class': 'emotion',
    'emotion': 'emotion',
    'emotion class': 'emotion',
    '\u044d\u043c\u043e\u0446\u0438\u044f': 'emotion',
    'source category': 'source_category',
    'source_category': 'source_category',
    '\u043a\u0430\u0442\u0435\u0433\u043e\u0440\u0438\u044f \u0438\u0441\u0442\u043e\u0447\u043d\u0438\u043a\u043e\u0432': 'source_category',
    'source count': 'source_count',
    'source_count': 'source_count',
    'occurrence count': 'source_count',
    '\u043a\u043e\u043b\u0438\u0447\u0435\u0441\u0442\u0432\u043e \u0432\u0445\u043e\u0436\u0434\u0435\u043d\u0438\u0439': 'source_count',
    'frequency of occurrence across sources': 'source_count',
    'sources': 'sources',
    '\u0438\u0441\u0442\u043e\u0447\u043d\u0438\u043a\u0438': 'sources',
    'source names': 'sources',
}

POSITIONAL_COLUMNS = (
    'term',
    'part_of_speech',
    'emotion',
    'source_category',
    'source_count',
    'sources',
)

WORD_RE = re.compile(r"[\u0430-\u044f\u0451]+(?:-[\u0430-\u044f\u0451]+)?", re.IGNORECASE)
TOKEN_RE = WORD_RE
POSITIVE_EMOTIONS = {'joy', 'surprise'}
NEGATIVE_EMOTIONS = {'sadness', 'fear', 'anger', 'disgust'}


def normalize_text(value: str, collapse_yo: bool = False) -> str:
    text = ' '.join(value.strip().lower().split())
    if collapse_yo:
        text = text.replace('\u0451', '\u0435')
    return text


def normalize_word(word: str, collapse_yo: bool = True) -> str:
    normalized = word.strip().lower()
    if collapse_yo:
        normalized = normalized.replace('\u0451', '\u0435')
    return normalized


def _normalize_header(value: str) -> str:
    return ' '.join(value.strip().lower().replace('_', ' ').split())


def _guess_delimiter(first_line: str) -> str:
    candidates = (';', '\t', ',')
    counts = {delimiter: first_line.count(delimiter) for delimiter in candidates}
    return max(counts, key=counts.get) if any(counts.values()) else ';'


def word_trigrams(word: str) -> set[str]:
    padded = f'^{word}$'
    if len(padded) < 3:
        return {padded}
    return {padded[index:index + 3] for index in range(len(padded) - 2)}


def levenshtein_distance(left: str, right: str) -> int:
    if left == right:
        return 0
    if not left:
        return len(right)
    if not right:
        return len(left)

    previous = list(range(len(right) + 1))
    for row_index, left_char in enumerate(left, start=1):
        current = [row_index]
        for col_index, right_char in enumerate(right, start=1):
            substitution_cost = 0 if left_char == right_char else 1
            current.append(
                min(
                    previous[col_index] + 1,
                    current[col_index - 1] + 1,
                    previous[col_index - 1] + substitution_cost,
                )
            )
        previous = current
    return previous[-1]


def build_word_frequencies_from_corpus(corpus_path: str | Path, *, min_word_len: int = 3, collapse_yo: bool = True) -> Counter[str]:
    path = Path(corpus_path)
    counter: Counter[str] = Counter()
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            for match in WORD_RE.findall(line):
                word = normalize_word(match, collapse_yo=collapse_yo)
                if len(word) >= min_word_len:
                    counter[word] += 1
    return counter


def save_word_frequencies(counter: Counter[str], output_path: str | Path) -> None:
    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='\n') as handle:
        for word, frequency in counter.most_common():
            handle.write(f'{word}\t{frequency}\n')


@dataclass(frozen=True, slots=True)
class CorrectionCandidate:
    word: str
    score: float
    frequency: int
    edit_distance: int
    jaccard: float


class TrigramSpellCorrector:
    def __init__(self, word_frequencies: Counter[str]) -> None:
        self.word_frequencies = Counter({normalize_word(word): freq for word, freq in word_frequencies.items() if freq > 0})
        self.trigram_index: dict[str, set[str]] = defaultdict(set)
        self.word_to_trigrams: dict[str, set[str]] = {}
        self.max_frequency = max(self.word_frequencies.values(), default=1)
        for word in self.word_frequencies:
            trigrams = word_trigrams(word)
            self.word_to_trigrams[word] = trigrams
            for trigram in trigrams:
                self.trigram_index[trigram].add(word)

    @classmethod
    def from_frequency_file(cls, path: str | Path) -> 'TrigramSpellCorrector':
        counter: Counter[str] = Counter()
        with Path(path).open('r', encoding='utf-8') as handle:
            for line in handle:
                stripped = line.strip()
                if not stripped:
                    continue
                parts = stripped.split()
                if len(parts) < 2:
                    continue
                try:
                    counter[normalize_word(parts[0])] = int(parts[1])
                except ValueError:
                    continue
        return cls(counter)

    def suggest(self, token: str, *, top_k: int = 5, max_edit_distance: int = 2, min_jaccard: float = 0.2) -> list[CorrectionCandidate]:
        normalized = normalize_word(token)
        if not normalized:
            return []
        if normalized in self.word_frequencies:
            return [CorrectionCandidate(normalized, 1.0, self.word_frequencies[normalized], 0, 1.0)]

        token_trigrams = word_trigrams(normalized)
        candidate_words: set[str] = set()
        for trigram in token_trigrams:
            candidate_words.update(self.trigram_index.get(trigram, set()))

        scored_candidates: list[CorrectionCandidate] = []
        for candidate in candidate_words:
            candidate_trigrams = self.word_to_trigrams[candidate]
            intersection = len(token_trigrams & candidate_trigrams)
            union = len(token_trigrams | candidate_trigrams)
            jaccard = intersection / union if union else 0.0
            if jaccard < min_jaccard:
                continue
            distance = levenshtein_distance(normalized, candidate)
            if distance > max_edit_distance:
                continue
            frequency_bonus = math.log1p(self.word_frequencies[candidate]) / math.log1p(self.max_frequency)
            score = (jaccard * 0.7) + (frequency_bonus * 0.2) + ((max_edit_distance - distance + 1) * 0.1)
            scored_candidates.append(CorrectionCandidate(candidate, score, self.word_frequencies[candidate], distance, jaccard))

        scored_candidates.sort(key=lambda item: (-item.score, item.edit_distance, -item.frequency, item.word))
        return scored_candidates[:top_k]

    def correct_token(self, token: str, *, min_score: float = 0.45, top_k: int = 5) -> str:
        suggestions = self.suggest(token, top_k=top_k)
        if not suggestions:
            return token
        return suggestions[0].word if suggestions[0].score >= min_score else token


@dataclass(frozen=True, slots=True)
class LexiconEntry:
    term: str
    emotion: str
    part_of_speech: str = ''
    source_category: str = ''
    source_count: int = 1
    sources: tuple[str, ...] = ()

    @property
    def is_phrase(self) -> bool:
        return ' ' in self.term


class EmotionLexicon:
    def __init__(self, entries, collapse_yo: bool = True) -> None:
        self.collapse_yo = collapse_yo
        self.entries_by_term: dict[str, list[LexiconEntry]] = {}
        for entry in entries:
            key = normalize_text(entry.term, collapse_yo=collapse_yo)
            self.entries_by_term.setdefault(key, []).append(entry)

    def lookup(self, term: str) -> list[LexiconEntry]:
        key = normalize_text(term, collapse_yo=self.collapse_yo)
        return self.entries_by_term.get(key, [])

    @property
    def max_phrase_len(self) -> int:
        if not self.entries_by_term:
            return 1
        return max(len(term.split()) for term in self.entries_by_term)


def _read_rows(path: Path) -> list[dict[str, str]]:
    text = path.read_text(encoding='utf-8-sig').replace('\r\n', '\n').replace('\r', '\n')
    lines = [line for line in text.split('\n') if line.strip()]
    if not lines:
        return []

    delimiter = _guess_delimiter(lines[0])
    reader = csv.reader(lines, delimiter=delimiter)
    rows = list(reader)
    if not rows:
        return []

    raw_headers = rows[0]
    normalized_headers = [_normalize_header(header) for header in raw_headers]
    canonical_headers = [HEADER_ALIASES.get(header, '') for header in normalized_headers]
    has_header = canonical_headers.count('term') == 1 and canonical_headers.count('emotion') == 1
    if not has_header:
        canonical_headers = list(POSITIONAL_COLUMNS[:len(raw_headers)])
        data_rows = rows
    else:
        data_rows = rows[1:]

    normalized_rows: list[dict[str, str]] = []
    for row in data_rows:
        if not any(cell.strip() for cell in row):
            continue
        padded_row = row + [''] * (len(canonical_headers) - len(row))
        normalized_rows.append(
            {
                column_name: padded_row[index].strip()
                for index, column_name in enumerate(canonical_headers)
                if column_name
            }
        )
    return normalized_rows


def load_rusemolex(path: str | Path, collapse_yo: bool = True) -> EmotionLexicon:
    rows = _read_rows(Path(path))
    entries = []
    for row in rows:
        term = row.get('term', '').strip()
        if not term:
            continue
        emotion_key = normalize_text(row.get('emotion', ''), collapse_yo=False)
        emotion = EMOTION_LABEL_MAP.get(emotion_key, emotion_key or 'unassigned')
        try:
            source_count = int(row.get('source_count', '').strip())
        except ValueError:
            source_count = 1
        sources_raw = row.get('sources', '').strip()
        if '|' in sources_raw:
            sources = tuple(part.strip() for part in sources_raw.split('|') if part.strip())
        elif ';' in sources_raw:
            sources = tuple(part.strip() for part in sources_raw.split(';') if part.strip())
        else:
            sources = tuple(part.strip() for part in sources_raw.split(',') if part.strip())
        entries.append(
            LexiconEntry(
                term=normalize_text(term, collapse_yo=collapse_yo),
                emotion=emotion,
                part_of_speech=row.get('part_of_speech', '').strip(),
                source_category=row.get('source_category', '').strip(),
                source_count=max(1, source_count),
                sources=sources,
            )
        )
    return EmotionLexicon(entries, collapse_yo=collapse_yo)


class Pymorphy3Lemmatizer:
    def __init__(self, collapse_yo: bool = True) -> None:
        from pymorphy3 import MorphAnalyzer
        self._morph = MorphAnalyzer()
        self._collapse_yo = collapse_yo

    def lemmatize_candidates(self, token: str) -> list[str]:
        normalized = normalize_text(token, collapse_yo=self._collapse_yo)
        if not normalized:
            return [normalized]

        parsed = self._morph.parse(normalized)
        candidates = [normalized]
        for parse in parsed:
            lemma = parse.normal_form
            if self._collapse_yo:
                lemma = lemma.replace('ё', 'е')
            if lemma not in candidates:
                candidates.append(lemma)
        return candidates

    def lemmatize(self, token: str) -> str:
        candidates = self.lemmatize_candidates(token)
        return candidates[1] if len(candidates) > 1 else candidates[0]


class NotebookDemoLemmatizer:
    MAPPING = {
        'очен': 'очень',
        'очень': 'очень',
        'рад': 'радость',
        'радость': 'радость',
        'счастье': 'счастье',
        'грустно': 'грусть',
        'грусть': 'грусть',
        'страшно': 'страх',
        'страх': 'страх',
        'зол': 'злость',
        'злость': 'злость',
        'спокоен': 'спокойный',
        'спокойный': 'спокойный',
        'ужасный': 'ужасный',
        'удивление': 'удивление',
    }

    def lemmatize_candidates(self, token: str) -> list[str]:
        lemma = self.MAPPING.get(token, token)
        return [token] if lemma == token else [token, lemma]

    def lemmatize(self, token: str) -> str:
        return self.MAPPING.get(token, token)


@dataclass(frozen=True, slots=True)
class MatchExplanation:
    original_token: str
    normalized_token: str
    corrected_token: str
    lemma: str
    matched_term: str
    emotion: str
    weight: float
    negated: bool
    intensified: bool
    source_category: str
    source_count: int


@dataclass(frozen=True, slots=True)
class AnalysisResult:
    original_text: str
    normalized_text: str
    corrected_text: str
    content_token_count: int
    enough_text: bool
    emotion_scores: dict[str, float]
    polarity_score: float
    dominant_emotion: str | None
    matched: tuple[MatchExplanation, ...] = ()


@dataclass(slots=True)
class AnalyzerConfig:
    min_content_tokens: int = 5
    correction_min_length: int = 4
    negation_window: int = 3
    intensifier_window: int = 1
    contrast_boost: float = 1.25
    pre_contrast_discount: float = 0.8
    intensity_multiplier: float = 1.5
    downtone_multiplier: float = 0.65
    negation_multiplier: float = -0.75
    collapse_yo: bool = True
    negations: frozenset[str] = field(default_factory=lambda: frozenset({'\u043d\u0435', '\u043d\u0435\u0442', '\u043d\u0438', '\u043d\u0438\u043a\u043e\u0433\u0434\u0430', '\u043d\u0438\u0447\u0443\u0442\u044c', '\u043d\u0438\u0441\u043a\u043e\u043b\u044c\u043a\u043e', '\u0441\u043e\u0432\u0441\u0435\u043c_\u043d\u0435', '\u0432\u043e\u0432\u0441\u0435_\u043d\u0435'}))
    intensifiers: frozenset[str] = field(default_factory=lambda: frozenset({'\u043e\u0447\u0435\u043d\u044c', '\u043a\u0440\u0430\u0439\u043d\u0435', '\u0441\u0438\u043b\u044c\u043d\u043e', '\u0443\u0436\u0430\u0441\u043d\u043e', '\u0441\u043e\u0432\u0441\u0435\u043c', '\u0447\u0440\u0435\u0437\u0432\u044b\u0447\u0430\u0439\u043d\u043e'}))
    downtoners: frozenset[str] = field(default_factory=lambda: frozenset({'\u043d\u0435\u043c\u043d\u043e\u0433\u043e', '\u0441\u043b\u0435\u0433\u043a\u0430', '\u0447\u0443\u0442\u044c', '\u0435\u0434\u0432\u0430', '\u043e\u0442\u0447\u0430\u0441\u0442\u0438'}))
    contrast_markers: frozenset[str] = field(default_factory=lambda: frozenset({'\u043d\u043e', '\u043e\u0434\u043d\u0430\u043a\u043e', '\u0437\u0430\u0442\u043e'}))


def _entry_weight(entry: LexiconEntry) -> float:
    category = entry.source_category.upper()
    category_bonus = {'A': 0.2, 'AB': 0.1, 'B': 0.0}.get(category, 0.0)
    count_bonus = min(max(entry.source_count, 1), 5) * 0.1
    return 1.0 + category_bonus + count_bonus


def _clause_multipliers(tokens: list[str], config: AnalyzerConfig) -> list[float]:
    if not any(token in config.contrast_markers for token in tokens):
        return [1.0] * len(tokens)
    multipliers = [1.0] * len(tokens)
    contrast_positions = [index for index, token in enumerate(tokens) if token in config.contrast_markers]
    first_contrast = contrast_positions[0]
    for index in range(first_contrast):
        multipliers[index] = config.pre_contrast_discount
    for contrast_index in contrast_positions:
        for index in range(contrast_index + 1, len(tokens)):
            multipliers[index] = config.contrast_boost
    return multipliers


class EmotionAnalyzer:
    def __init__(self, lexicon: EmotionLexicon, *, lemmatizer=None, corrector: TrigramSpellCorrector | None = None, config: AnalyzerConfig | None = None) -> None:
        self.lexicon = lexicon
        self.corrector = corrector
        self.config = config or AnalyzerConfig(collapse_yo=lexicon.collapse_yo)
        if lemmatizer is not None:
            self.lemmatizer = lemmatizer
        elif PYMORPHY3_AVAILABLE:
            self.lemmatizer = Pymorphy3Lemmatizer(collapse_yo=self.config.collapse_yo)
        else:
            self.lemmatizer = NotebookDemoLemmatizer()

    def _tokenize(self, text: str) -> list[str]:
        return [normalize_text(match.group(0), collapse_yo=self.config.collapse_yo) for match in TOKEN_RE.finditer(text)]

    def _correct_token(self, token: str) -> str:
        if not self.corrector or len(token) < self.config.correction_min_length:
            return token
        return self.corrector.correct_token(token)

    def _lemma_candidates(self, token: str) -> list[str]:
        if hasattr(self.lemmatizer, 'lemmatize_candidates'):
            raw_candidates = self.lemmatizer.lemmatize_candidates(token)
        else:
            raw_candidates = [self.lemmatizer.lemmatize(token)]

        candidates: list[str] = []
        for candidate in raw_candidates:
            normalized_candidate = normalize_text(candidate, collapse_yo=self.config.collapse_yo)
            if normalized_candidate and normalized_candidate not in candidates:
                candidates.append(normalized_candidate)

        normalized_token = normalize_text(token, collapse_yo=self.config.collapse_yo)
        if normalized_token and normalized_token not in candidates:
            candidates.insert(0, normalized_token)

        return candidates or [normalized_token]

    def _lookup_single_token(self, token: str):
        candidates = self._lemma_candidates(token)
        preferred_lemma = candidates[1] if len(candidates) > 1 else candidates[0]
        for candidate in candidates:
            entries = self.lexicon.lookup(candidate)
            if entries:
                return entries, candidate, candidates
        return [], preferred_lemma, candidates

    def _find_phrase_match(self, tokens: list[str], lemmas: list[str], start_index: int, consumed_until: int):
        if start_index < consumed_until:
            return consumed_until, None
        max_len = min(self.lexicon.max_phrase_len, len(tokens) - start_index)
        for phrase_len in range(max_len, 1, -1):
            surface_candidate = ' '.join(tokens[start_index:start_index + phrase_len])
            entries = self.lexicon.lookup(surface_candidate)
            if entries:
                return start_index + phrase_len, entries[0]
            lemma_candidate = ' '.join(lemmas[start_index:start_index + phrase_len])
            entries = self.lexicon.lookup(lemma_candidate)
            if entries:
                return start_index + phrase_len, entries[0]
        return start_index + 1, None

    def analyze(self, text: str) -> AnalysisResult:
        raw_tokens = self._tokenize(text)
        corrected_tokens = [self._correct_token(token) for token in raw_tokens]
        lemma_candidates = [self._lemma_candidates(token) for token in corrected_tokens]
        lemmas = [candidates[1] if len(candidates) > 1 else candidates[0] for candidates in lemma_candidates]
        clause_multipliers = _clause_multipliers(corrected_tokens, self.config)
        enough_text = len(corrected_tokens) >= self.config.min_content_tokens
        emotion_scores: dict[str, float] = {}
        matched: list[MatchExplanation] = []
        consumed_until = 0

        for index, token in enumerate(corrected_tokens):
            next_index, phrase_entry = self._find_phrase_match(corrected_tokens, lemmas, index, consumed_until)
            if phrase_entry is not None:
                weight = _entry_weight(phrase_entry) * clause_multipliers[index]
                emotion_scores[phrase_entry.emotion] = emotion_scores.get(phrase_entry.emotion, 0.0) + weight
                matched.append(MatchExplanation(' '.join(raw_tokens[index:next_index]), ' '.join(corrected_tokens[index:next_index]), ' '.join(corrected_tokens[index:next_index]), ' '.join(lemmas[index:next_index]), phrase_entry.term, phrase_entry.emotion, weight, False, False, phrase_entry.source_category, phrase_entry.source_count))
                consumed_until = next_index
                continue

            if index < consumed_until:
                continue

            entries, matched_lemma, _ = self._lookup_single_token(token)
            if not entries:
                continue
            entry = entries[0]
            weight = _entry_weight(entry) * clause_multipliers[index]
            left_context = lemmas[max(0, index - self.config.negation_window):index]
            short_context = lemmas[max(0, index - self.config.intensifier_window):index]
            negated = any(context_token in self.config.negations for context_token in left_context)
            intensified = any(context_token in self.config.intensifiers for context_token in short_context)
            down_toned = any(context_token in self.config.downtoners for context_token in short_context)
            if intensified:
                weight *= self.config.intensity_multiplier
            if down_toned:
                weight *= self.config.downtone_multiplier
            if negated:
                weight *= self.config.negation_multiplier
            emotion_scores[entry.emotion] = emotion_scores.get(entry.emotion, 0.0) + weight
            matched.append(MatchExplanation(raw_tokens[index], token, token, matched_lemma, entry.term, entry.emotion, weight, negated, intensified, entry.source_category, entry.source_count))

        positive_score = sum(score for emotion, score in emotion_scores.items() if emotion in POSITIVE_EMOTIONS)
        negative_score = sum(score for emotion, score in emotion_scores.items() if emotion in NEGATIVE_EMOTIONS)
        polarity_score = positive_score - negative_score
        dominant_emotion = None
        if emotion_scores:
            top_emotion, top_score = max(emotion_scores.items(), key=lambda item: item[1])
            if top_score > 0:
                dominant_emotion = top_emotion
        return AnalysisResult(
            original_text=text,
            normalized_text=' '.join(raw_tokens),
            corrected_text=' '.join(corrected_tokens),
            content_token_count=len(corrected_tokens),
            enough_text=enough_text,
            emotion_scores={emotion: round(score, 4) for emotion, score in sorted(emotion_scores.items())},
            polarity_score=round(polarity_score, 4),
            dominant_emotion=dominant_emotion,
            matched=tuple(matched),
        )


## Build or load frequency table

Preferred order:
1. Load cached `wordfreq_ru.tsv`
2. Build it from the installed `wordfreq` package
3. Fall back to a tiny demo vocabulary


In [ ]:
corrector = None

def build_word_frequencies_from_wordfreq(*, lang: str = 'ru', wordlist: str = 'large', max_words: int = 200_000, min_zipf: float = 2.5, min_word_len: int = 3) -> Counter[str]:
    if not WORDFREQ_AVAILABLE:
        raise RuntimeError('wordfreq is not installed')

    counter: Counter[str] = Counter()
    added = 0
    for word in iter_wordlist(lang, wordlist=wordlist):
        normalized = normalize_word(word)
        if len(normalized) < min_word_len:
            continue
        if not WORD_RE.fullmatch(normalized):
            continue
        zipf = zipf_frequency(normalized, lang, wordlist=wordlist)
        if zipf < min_zipf:
            continue
        counter[normalized] = max(1, int(round(10 ** zipf)))
        added += 1
        if added >= max_words:
            break
    return counter

if WORDFREQ_FREQ_PATH.exists():
    corrector = TrigramSpellCorrector.from_frequency_file(WORDFREQ_FREQ_PATH)
    print(f'Loaded frequency table from {WORDFREQ_FREQ_PATH}')
elif WORDFREQ_AVAILABLE:
    word_frequencies = build_word_frequencies_from_wordfreq(
        lang=WORDFREQ_LANGUAGE,
        wordlist=WORDFREQ_WORDLIST,
        max_words=WORDFREQ_MAX_WORDS,
        min_zipf=WORDFREQ_MIN_ZIPF,
    )
    save_word_frequencies(word_frequencies, WORDFREQ_FREQ_PATH)
    corrector = TrigramSpellCorrector(word_frequencies)
    print(f'Built and saved {len(word_frequencies)} words from wordfreq to {WORDFREQ_FREQ_PATH}')
else:
    demo_frequencies = Counter({
        '\u044f': 500,
        '\u043c\u043d\u0435': 250,
        '\u044d\u0442\u043e': 300,
        '\u043e\u0447\u0435\u043d\u044c': 220,
        '\u043e\u0447\u0435\u043d': 5,
        '\u044d\u0442\u043e\u043c\u0443': 150,
        '\u0434\u043d\u044e': 120,
        '\u0447\u0443\u0432\u0441\u0442\u0432\u0443\u044e': 80,
        '\u0441\u0435\u0433\u043e\u0434\u043d\u044f': 170,
        '\u0440\u0435\u0437\u0443\u043b\u044c\u0442\u0430\u0442': 160,
        '\u0441\u043a\u043e\u0440\u0435\u0435': 140,
        '\u0441\u043b\u0435\u0433\u043a\u0430': 90,
        '\u0432': 400,
        '\u0446\u0435\u043b\u043e\u043c': 110,
        '\u0438': 600,
        '\u043d\u0435': 500,
        '\u043d\u043e': 450,
        '\u0440\u0430\u0434': 120,
        '\u0440\u0430\u0434\u043e\u0441\u0442\u044c': 100,
        '\u0441\u0447\u0430\u0441\u0442\u044c\u0435': 80,
        '\u0433\u0440\u0443\u0441\u0442\u043d\u043e': 90,
        '\u0433\u0440\u0443\u0441\u0442\u044c': 70,
        '\u0441\u0442\u0440\u0430\u0448\u043d\u043e': 85,
        '\u0441\u0442\u0440\u0430\u0445': 60,
        '\u0437\u043e\u043b': 75,
        '\u0437\u043b\u043e\u0441\u0442\u044c': 50,
        '\u0443\u0434\u0438\u0432\u043b\u0435\u043d\u0438\u0435': 40,
        '\u0441\u043f\u043e\u043a\u043e\u0435\u043d': 65,
        '\u0441\u043f\u043e\u043a\u043e\u0439\u043d\u044b\u0439': 25,
        '\u0443\u0436\u0430\u0441\u043d\u044b\u0439': 20,
        '\u043f\u0440\u0435\u043a\u0440\u0430\u0441\u043d\u044b\u0439': 20,
    })
    corrector = TrigramSpellCorrector(demo_frequencies)
    print('wordfreq is not available. Using a tiny demo vocabulary instead.')

print('Vocabulary size:', len(corrector.word_frequencies))


## Load RusEmoLex and initialize analyzer


In [ ]:
if LEXICON_PATH.exists():
    lexicon = load_rusemolex(LEXICON_PATH)
    print(f'Loaded RusEmoLex from {LEXICON_PATH}')
else:
    lexicon = EmotionLexicon([
        LexiconEntry(term='\u0440\u0430\u0434\u043e\u0441\u0442\u044c', emotion='joy', source_category='A', source_count=4),
        LexiconEntry(term='\u0441\u0447\u0430\u0441\u0442\u044c\u0435', emotion='joy', source_category='A', source_count=3),
        LexiconEntry(term='\u0433\u0440\u0443\u0441\u0442\u044c', emotion='sadness', source_category='AB', source_count=2),
        LexiconEntry(term='\u0441\u0442\u0440\u0430\u0445', emotion='fear', source_category='A', source_count=3),
        LexiconEntry(term='\u0437\u043b\u043e\u0441\u0442\u044c', emotion='anger', source_category='A', source_count=3),
        LexiconEntry(term='\u0443\u0436\u0430\u0441\u043d\u044b\u0439', emotion='disgust', source_category='AB', source_count=2),
        LexiconEntry(term='\u0443\u0434\u0438\u0432\u043b\u0435\u043d\u0438\u0435', emotion='surprise', source_category='AB', source_count=2),
        LexiconEntry(term='\u0441\u043f\u043e\u043a\u043e\u0439\u043d\u044b\u0439', emotion='joy', source_category='AB', source_count=1),
    ])
    print('RusEmoLex.csv not found. Using a tiny demo lexicon instead.')

print('Lexicon terms:', len(lexicon.entries_by_term))

if PYMORPHY3_AVAILABLE:
    analyzer = EmotionAnalyzer(lexicon, lemmatizer=Pymorphy3Lemmatizer(), corrector=corrector)
    print('Analyzer ready with pymorphy3 lemmatization')
else:
    analyzer = EmotionAnalyzer(lexicon, lemmatizer=NotebookDemoLemmatizer(), corrector=corrector)
    print('Analyzer ready with demo fallback lemmatizer')


## Emotion distribution and plotting

The notebook keeps raw `emotion_scores`, but the final user-facing output is a normalized distribution over emotions.
The plot shows per-word contributions for each emotion, and the dominant emotion is highlighted with a thicker line.


In [ ]:
import matplotlib.pyplot as plt

EMOTION_ORDER = ['joy', 'sadness', 'fear', 'anger', 'surprise', 'disgust']
EMOTION_COLORS = {
    'joy': '#f4b400',
    'sadness': '#4285f4',
    'fear': '#7e57c2',
    'anger': '#db4437',
    'surprise': '#0f9d58',
    'disgust': '#8d6e63',
}


def build_emotion_distribution(result: AnalysisResult) -> tuple[dict[str, float], dict[str, float], str | None]:
    positive_scores = {
        emotion: max(result.emotion_scores.get(emotion, 0.0), 0.0)
        for emotion in EMOTION_ORDER
    }
    total = sum(positive_scores.values())
    distribution = {
        emotion: round((positive_scores[emotion] / total) if total else 0.0, 4)
        for emotion in EMOTION_ORDER
    }
    dominant = max(distribution, key=distribution.get) if total else None
    return positive_scores, distribution, dominant


def build_emotion_trace(result: AnalysisResult) -> tuple[list[str], dict[str, list[float]]]:
    tokens = result.corrected_text.split()
    traces = {emotion: [0.0] * len(tokens) for emotion in EMOTION_ORDER}

    if not tokens:
        return tokens, traces

    cursor = 0
    for item in result.matched:
        span_tokens = item.corrected_token.split()
        if not span_tokens:
            continue

        span_len = len(span_tokens)
        found_start = None

        for start in range(cursor, len(tokens) - span_len + 1):
            if tokens[start:start + span_len] == span_tokens:
                found_start = start
                break

        if found_start is None:
            for start in range(0, len(tokens) - span_len + 1):
                if tokens[start:start + span_len] == span_tokens:
                    found_start = start
                    break

        if found_start is None:
            continue

        share = item.weight / span_len
        for offset in range(span_len):
            traces[item.emotion][found_start + offset] += share

        cursor = found_start + span_len

    return tokens, traces


def plot_emotion_trace(result: AnalysisResult, *, title: str | None = None) -> None:
    _, distribution, dominant = build_emotion_distribution(result)
    tokens, traces = build_emotion_trace(result)

    if not tokens:
        print('No tokens to plot')
        return

    x = list(range(1, len(tokens) + 1))
    fig_width = max(10, len(tokens) * 0.8)
    plt.figure(figsize=(fig_width, 5))

    for emotion in EMOTION_ORDER:
        linewidth = 3.6 if emotion == dominant else 1.8
        alpha = 1.0 if emotion == dominant else 0.75
        plt.plot(
            x,
            traces[emotion],
            marker='o',
            linewidth=linewidth,
            alpha=alpha,
            color=EMOTION_COLORS[emotion],
            label=f"{emotion} ({distribution[emotion]:.2f})",
        )

    plt.axhline(0, color='black', linewidth=0.8, alpha=0.4)
    plt.xticks(x, [f'{idx}:{token}' for idx, token in zip(x, tokens)], rotation=45, ha='right')
    plt.xlabel('Word ID')
    plt.ylabel('Emotion contribution')
    plt.title(title or f'Emotion trace | dominant={dominant or "none"}')
    legend = plt.legend(title='Emotion distribution')

    if dominant is not None:
        for text in legend.get_texts():
            if text.get_text().startswith(dominant):
                text.set_fontweight('bold')

    plt.tight_layout()
    plt.show()


## End-to-end demo on OCR-like text


In [ ]:
examples = [
    '\u044f \u043e\u0447\u0435\u043d \u0440\u0430\u0434 \u044d\u0442\u043e\u043c\u0443 \u0434\u043d\u044e \u0438 \u0447\u0443\u0432\u0441\u0442\u0432\u0443\u044e \u0441\u0447\u0430\u0441\u0442\u044c\u0435',
    '\u043c\u043d\u0435 \u043e\u0447\u0435\u043d\u044c \u0433\u0440\u0443\u0441\u0442\u043d\u043e \u0438 \u0441\u0442\u0440\u0430\u0448\u043d\u043e \u0441\u0435\u0433\u043e\u0434\u043d\u044f',
    '\u044d\u0442\u043e \u043d\u0435 \u0443\u0436\u0430\u0441\u043d\u044b\u0439 \u0440\u0435\u0437\u0443\u043b\u044c\u0442\u0430\u0442, \u0430 \u0441\u043a\u043e\u0440\u0435\u0435 \u0443\u0434\u0438\u0432\u043b\u0435\u043d\u0438\u0435 \u0438 \u0440\u0430\u0434\u043e\u0441\u0442\u044c',
    '\u044f \u0441\u043b\u0435\u0433\u043a\u0430 \u0437\u043e\u043b, \u043d\u043e \u0432 \u0446\u0435\u043b\u043e\u043c \u0441\u043f\u043e\u043a\u043e\u0435\u043d',
]

results = []
for text in examples:
    result = analyzer.analyze(text)
    results.append(result)
    _, distribution, dominant = build_emotion_distribution(result)
    print('=' * 80)
    print('INPUT:        ', result.original_text)
    print('NORMALIZED:   ', result.normalized_text)
    print('CORRECTED:    ', result.corrected_text)
    print('TOKENS:       ', result.content_token_count)
    print('ENOUGH TEXT:  ', result.enough_text)
    print('RAW SCORES:   ', result.emotion_scores)
    print('DISTRIBUTION: ', distribution)
    print('DOMINANT:     ', dominant)


## Emotion graph

Each colored line shows how strongly a specific emotion is expressed across the recognized words.
The dominant emotion is drawn with a thicker line and appears bold in the legend.


In [ ]:
result = results[0]
plot_emotion_trace(result, title='Example 1 emotion trace')


## Inspect token-level matches


In [ ]:
result = results[0]
for item in result.matched:
    print({
        'original': item.original_token,
        'corrected': item.corrected_token,
        'lemma': item.lemma,
        'matched_term': item.matched_term,
        'emotion': item.emotion,
        'weight': item.weight,
        'negated': item.negated,
        'intensified': item.intensified,
    })


## Recommended next step

Run the notebook on real OCR output and compare the resulting emotion distributions and token-level traces for different samples.
